In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import glob

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 34
PROJECT_ROOT: C:\Users\mjbou\governance-framework


## IMF Fiscal Rules Pipeline

**Source:** IMF Fiscal Affairs Department — Fiscal Rules Dataset
**Access:** Manual Excel download (auto-detects file in Downloads)
**Download instructions:** See `docs/instructions_data_maintenance.md` — IMF_FISCAL_RULES section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| Presence of fiscal rules (budget balance, debt, expenditure, revenue) | Macroeconomic policy framework quality | Primary tier 1 |

In [3]:
import pandas as pd
import os
import glob
from datetime import datetime

# Auto-detect IMF Fiscal Rules Excel in Downloads — no hardcoded filename
fr_pattern = os.path.join(DOWNLOADS_DIR, "*Fiscal Rules*.xlsx")
fr_files = glob.glob(fr_pattern)

if not fr_files:
    print(f"No IMF Fiscal Rules file found in {DOWNLOADS_DIR}")
    print("Download from imf.org/en/topics/fiscal-policies/fiscal-rules-dataset")
else:
    # Use most recent download
    fr_file = max(fr_files, key=os.path.getmtime)
    print(f"Found: {os.path.basename(fr_file)}")
    
    # Inspect sheets
    xl = pd.ExcelFile(fr_file, engine='openpyxl')
    print(f"Sheets: {xl.sheet_names}")

Found: Publication - IMF FAD Fiscal Rules Dataset 1985-2024 Update.xlsx
Sheets: ['Cover', 'Abbreviation and codes', 'Revisions NEW', 'Rules', 'Supranational', 'European Union', 'EU ', 'ECCU', 'EAMU (EAC)', 'CEMAC', 'WEAMU', 'FAD own checking', 'WEO_IFS', 'weo_group']


In [4]:
# Inspect the Rules sheet — the main national fiscal rules data
# Try reading with no header first to see the layout
rules_preview = pd.read_excel(fr_file, sheet_name='Rules', engine='openpyxl', header=None, nrows=5)
print("First 5 rows (no header):")
print(rules_preview.to_string())
print(f"\nFull sheet shape: {pd.read_excel(fr_file, sheet_name='Rules', engine='openpyxl', header=None).shape}")

First 5 rows (no header):
         0     1             2                             3                 4                         5              6                                          7                 8                         9              10                      11                12                        13             14                                                       15                16                        17             18                                                      19         20                 21                                                                                22                                           23   24   25   26                   27   28   29   30                             31   32   33   34                   35   36   37   38              39   40   41   42                   43   44   45   46                                       47   48   49   50                   51   52   53   54              55   56   57   58                 

In [5]:
# Load Rules sheet with data starting after the multi-row header
# Header rows span 0-3; data begins row 4. Use header=None then select by column position.
rules_raw = pd.read_excel(fr_file, sheet_name='Rules', engine='openpyxl', header=None, skiprows=4)

# Select key columns by position (stable across vintages — same template structure):
# 1=year, 2=country name, 3-6=rule type in place (ER/RR/BBR/DR), 113=ifscode, 114=ccode
col_map = {
    1:   'year',
    2:   'country_name',
    3:   'fr_expenditure_rule',
    4:   'fr_revenue_rule',
    5:   'fr_budget_balance_rule',
    6:   'fr_debt_rule',
    113: 'ifs_code',
    114: 'country_code',
}

fr = rules_raw[list(col_map.keys())].copy()
fr.columns = list(col_map.values())

# Drop rows with no year or country
fr = fr[fr['year'].notna() & fr['country_name'].notna()].copy()
fr['year'] = pd.to_numeric(fr['year'], errors='coerce').astype('Int64')

print(f"Shape: {fr.shape}")
print(f"Years: {fr['year'].min()} — {fr['year'].max()}")
print(f"Countries: {fr['country_name'].nunique()}")
print(f"\nSample rule values:")
print(fr['fr_budget_balance_rule'].value_counts().head())
print(fr.head(3).to_string())

Shape: (4920, 8)
Years: 1985 — 2024
Countries: 123

Sample rule values:
fr_budget_balance_rule
-    2725
1    2195
Name: count, dtype: int64
   year country_name fr_expenditure_rule fr_revenue_rule fr_budget_balance_rule fr_debt_rule  ifs_code country_code
0  1985      Andorra                   -               -                      -            -       171          AND
1  1986      Andorra                   -               -                      -            -       171          AND
2  1987      Andorra                   -               -                      -            -       171          AND


In [6]:
# Convert rule indicators from "1"/"-" to binary 1/0
# "1" = rule in place, "-" = no rule
rule_cols = ['fr_expenditure_rule', 'fr_revenue_rule', 'fr_budget_balance_rule', 'fr_debt_rule']
for col in rule_cols:
    fr[col] = (fr[col].astype(str).str.strip() == '1').astype(int)

# Derive a count of total rule types in place per country-year
fr['fr_num_rule_types'] = fr[rule_cols].sum(axis=1)

# Filter to framework start year
fr = fr[fr['year'] >= FRAMEWORK_START_YEAR].copy()
fr = fr.sort_values(['country_name', 'year']).reset_index(drop=True)

print(f"Shape: {fr.shape}")
print(f"Years: {fr['year'].min()} — {fr['year'].max()}")
print(f"Countries: {fr['country_name'].nunique()}")
print(f"\nRule type prevalence (country-years with each rule):")
for col in rule_cols:
    print(f"  {col}: {fr[col].sum()}")
print(f"\nMissing values: {fr.isnull().sum().sum()}")
print(fr.head(3).to_string())


Shape: (4305, 9)
Years: 1990 — 2024
Countries: 123

Rule type prevalence (country-years with each rule):
  fr_expenditure_rule: 935
  fr_revenue_rule: 406
  fr_budget_balance_rule: 2160
  fr_debt_rule: 1926

Missing values: 0
   year country_name  fr_expenditure_rule  fr_revenue_rule  fr_budget_balance_rule  fr_debt_rule  ifs_code country_code  fr_num_rule_types
0  1990      Andorra                    0                0                       0             0       171          AND                  0
1  1991      Andorra                    0                0                       0             0       171          AND                  0
2  1992      Andorra                    0                0                       0             0       171          AND                  0


In [7]:
# Derive metadata from data — no hardcoding
latest_year = str(int(fr['year'].max()))

# Save to processed
output_path = os.path.join(PROCESSED_DIR, "imf_fiscal_rules_clean.csv")
fr.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {fr.shape}")

# Update download log
update_entry(
    "IMF_FISCAL_RULES",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="imf_fiscal_rules_clean.csv",
    latest_available_version=latest_year,
    notes="Presence of four fiscal rule types (expenditure, revenue, budget balance, debt) as binary, plus count. "
          "Manual Excel download — pipeline auto-detects file in Downloads. "
          "Columns selected by position due to 4-row nested header — if IMF restructures template, column mapping needs review. "
          "Coverage: 123 countries."
)
print_entry("IMF_FISCAL_RULES")

Written: C:\Users\mjbou\governance-framework\data\processed\imf_fiscal_rules_clean.csv
Shape: (4305, 9)
[download_log] Updated entry for IMF_FISCAL_RULES
  source_id: IMF_FISCAL_RULES
  last_attempted_date: 2026-06-17
  last_successful_download_date: 2026-06-17
  data_as_of_date: 2024
  local_filename: imf_fiscal_rules_clean.csv
  latest_available_version: 2024
  no_update_reason: nan
  notes: Presence of four fiscal rule types (expenditure, revenue, budget balance, debt) as binary, plus count. Manual Excel download — pipeline auto-detects file in Downloads. Columns selected by position due to 4-row nested header — if IMF restructures template, column mapping needs review. Coverage: 123 countries.
